# Two-hour pilot, decoding diagnostics, and joint training

Historical research source, original zero-based cells **56?62**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 56

Position 57 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 52 - TWO-HOUR TRAINING DATASET
#
# Goal: approximately 2 hours of diverse Amharic audio.
# Preserve existing evaluation speaker separation.
# Exclude known quarantined recordings.
#
# No GPU training. No checkpoint changes.
# ============================================================

import json
import random
import hashlib
import statistics
from pathlib import Path
from collections import defaultdict, Counter
from IPython.display import Audio, display
from transformers import Wav2Vec2Config

ROOT_52 = DATA_DIR / "scaled_amharic_64_16_v1"
OUT_52 = DATA_DIR / "amharic_2hour_pilot_v1"
OUT_52.mkdir(parents=True, exist_ok=True)

SELECTION_52 = OUT_52 / "selection.json"
SEED_52 = 20261020
TARGET_SECONDS_52 = 7200
MAX_RECORDINGS_52 = 750
MAX_PER_SPEAKER_52 = 5

clean = json.loads(
    (ROOT_52 / "quality_filtered_v2.json").read_text(
        encoding="utf-8"
    )
)

original = json.loads(
    (ROOT_52 / "selection.json").read_text(
        encoding="utf-8"
    )
)

records_52 = {r["id"]: r for r in train_records}

# ------------------------------------------------------------
# 1. Protect evaluation examples AND speakers
# ------------------------------------------------------------

protected_ids_52 = set(original["val_ids"])
protected_ids_52.update(clean["primary_val_ids"])
protected_ids_52.update(clean["noisy_challenge_ids"])

prior_validation_files = [
    DATA_DIR / "short_multirecording_head_only_v2"
    / "split.json",

    DATA_DIR / "head_only_multirecording_v1"
    / "selection.json"
]

for path in prior_validation_files:
    if path.is_file():
        previous = json.loads(
            path.read_text(encoding="utf-8")
        )
        protected_ids_52.update(
            previous.get("val_ids", [])
        )

# Protect the original fixed evaluation IDs/speakers
# too, if its metadata is present in the JSONL file.
external_speakers_52 = set()

external_path_52 = DATA_DIR / "baseline_eval_fixed_12.jsonl"

if external_path_52.is_file():
    for line in external_path_52.read_text(
        encoding="utf-8"
    ).splitlines():
        if not line.strip():
            continue

        entry = json.loads(line)
        rid = entry.get("id")

        if rid:
            protected_ids_52.add(rid)

        if entry.get("speaker_id"):
            external_speakers_52.add(
                entry["speaker_id"]
            )

protected_speakers_52 = {
    records_52[rid]["speaker_id"]
    for rid in protected_ids_52
    if rid in records_52
}
protected_speakers_52.update(external_speakers_52)

quarantined_52 = set(
    clean["quarantined_train_ids"]
)

print("PROTECTION CHECKS")
print("=" * 55)
print("Protected recording IDs:", len(protected_ids_52))
print("Protected speaker IDs:", len(protected_speakers_52))
print("Quarantined recordings:", len(quarantined_52))

# ------------------------------------------------------------
# 2. Verify recording eligibility
# ------------------------------------------------------------

cfg_52 = Wav2Vec2Config.from_pretrained(
    str(BASELINE_DIR)
)

def frames_52(n_samples):
    n = int(n_samples)
    for kernel, stride in zip(
        cfg_52.conv_kernel, cfg_52.conv_stride
    ):
        n = (n - kernel) // stride + 1
    return n

def eligible_52(record):
    rid = record["id"]
    sid = record.get("speaker_id")

    if not sid:
        return False
    if rid in protected_ids_52 | quarantined_52:
        return False
    if sid in protected_speakers_52:
        return False

    seconds = float(record["duration_seconds"])
    if not 4.0 <= seconds <= 18.0:
        return False

    text = record["transcription"]

    if any(ch.isdigit() for ch in text):
        return False

    audio_path = Path(AUDIO_DIR) / record["audio_file"]
    if not audio_path.is_file():
        return False

    labels = tokenizer.encode(
        text, add_special_tokens=False
    )

    if not 20 <= len(labels) <= 160:
        return False

    if tokenizer.unk_token_id in labels:
        return False
    if tokenizer.pad_token_id in labels:
        return False

    repeats = sum(
        a == b
        for a, b in zip(labels[:-1], labels[1:])
    )

    available = frames_52(round(seconds * 16000))
    return available - len(labels) - repeats >= 20

eligible_records_52 = [
    r for r in train_records
    if eligible_52(r)
]

print("\nEligible candidates:", len(eligible_records_52))
print(
    "Eligible speakers:",
    len({r["speaker_id"] for r in eligible_records_52})
)

# ------------------------------------------------------------
# 3. Retain valid previously trained examples
# ------------------------------------------------------------

carryover_52 = [
    rid for rid in clean["train_ids"]
    if eligible_52(records_52[rid])
]

excluded_carryover_52 = [
    rid for rid in clean["train_ids"]
    if rid not in carryover_52
]

print("\nPrevious training recordings retained:",
      len(carryover_52))

print("Additional carryover exclusions:",
      len(excluded_carryover_52))

# ------------------------------------------------------------
# 4. Speaker-balanced selection
# ------------------------------------------------------------

if SELECTION_52.exists():

    selection_52 = json.loads(
        SELECTION_52.read_text(encoding="utf-8")
    )

    selected_ids_52 = selection_52["train_ids"]

    print("\nRestored existing saved selection.")

else:

    rng = random.Random(SEED_52)

    selected_ids_52 = list(carryover_52)
    selected_set_52 = set(selected_ids_52)

    counts_52 = Counter(
        records_52[rid]["speaker_id"]
        for rid in selected_ids_52
    )

    total_seconds_52 = sum(
        float(records_52[rid]["duration_seconds"])
        for rid in selected_ids_52
    )

    by_speaker_52 = defaultdict(list)

    for record in eligible_records_52:
        if record["id"] not in selected_set_52:
            by_speaker_52[
                record["speaker_id"]
            ].append(record)

    speakers_52 = sorted(by_speaker_52)
    rng.shuffle(speakers_52)

    for sid in speakers_52:
        rng.shuffle(by_speaker_52[sid])

    # Round-robin sampling promotes speaker diversity.
    for _ in range(MAX_PER_SPEAKER_52):

        if total_seconds_52 >= TARGET_SECONDS_52:
            break

        for sid in speakers_52:

            if total_seconds_52 >= TARGET_SECONDS_52:
                break
            if len(selected_ids_52) >= MAX_RECORDINGS_52:
                break
            if counts_52[sid] >= MAX_PER_SPEAKER_52:
                continue
            if not by_speaker_52[sid]:
                continue

            record = by_speaker_52[sid].pop()
            rid = record["id"]

            selected_ids_52.append(rid)
            selected_set_52.add(rid)

            counts_52[sid] += 1
            total_seconds_52 += float(
                record["duration_seconds"]
            )

    assert total_seconds_52 >= TARGET_SECONDS_52, (
        "Unable to reach 2 hours under these constraints. "
        f"Selected {total_seconds_52 / 3600:.2f} hours."
    )

    selection_52 = {
        "seed": SEED_52,
        "source": "train_final_25h_ctc_safe",
        "train_ids": selected_ids_52,
        "carryover_ids": carryover_52,
        "excluded_carryover_ids": excluded_carryover_52,
        "primary_val_ids": clean["primary_val_ids"],
        "noisy_challenge_ids": clean["noisy_challenge_ids"],
        "quarantined_ids": sorted(quarantined_52),
        "target_audio_seconds": TARGET_SECONDS_52,
        "max_recordings": MAX_RECORDINGS_52,
        "max_per_speaker": MAX_PER_SPEAKER_52
    }

    SELECTION_52.write_text(
        json.dumps(
            selection_52,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("\nNew two-hour selection saved.")

# ------------------------------------------------------------
# 5. Check for leakage and invalid examples
# ------------------------------------------------------------

selected_records_52 = [
    records_52[rid]
    for rid in selected_ids_52
]

assert len(selected_ids_52) == len(set(selected_ids_52))

assert all(
    eligible_52(r)
    for r in selected_records_52
)

train_speakers_52 = {
    r["speaker_id"] for r in selected_records_52
}

assert not (
    train_speakers_52 & protected_speakers_52
)
assert not (
    set(selected_ids_52) & protected_ids_52
)
assert not (
    set(selected_ids_52) & quarantined_52
)

# ------------------------------------------------------------
# 6. Verify tokenizer reconstruction again
# ------------------------------------------------------------

roundtrip_errors_52 = []

for record in selected_records_52:

    original_text = normalize_amharic_ctc(
        record["transcription"]
    )

    ids = tokenizer.encode(
        record["transcription"],
        add_special_tokens=False
    )

    recovered = normalize_amharic_ctc(
        tokenizer.decode(
            ids, group_tokens=False
        )
    )

    if original_text != recovered:
        roundtrip_errors_52.append(record["id"])

assert not roundtrip_errors_52, (
    f"Tokenizer mismatches: {roundtrip_errors_52[:5]}"
)

# ------------------------------------------------------------
# 7. Summarize the selection
# ------------------------------------------------------------

durations_52 = [
    float(r["duration_seconds"])
    for r in selected_records_52
]

total_hours_52 = sum(durations_52) / 3600

speaker_counts_52 = Counter(
    r["speaker_id"] for r in selected_records_52
)

print("\n" + "=" * 60)
print("TWO-HOUR DATASET SUMMARY")
print("=" * 60)

print("Selected recordings:", len(selected_records_52))
print("Selected speakers:", len(train_speakers_52))
print("Total audio:", round(total_hours_52, 3), "hours")
print("Average duration:",
      round(statistics.mean(durations_52), 2), "seconds")
print("Shortest duration:", round(min(durations_52), 2))
print("Longest duration:", round(max(durations_52), 2))

print("Maximum recordings from one speaker:",
      max(speaker_counts_52.values()))

print("Tokenizer round-trip mismatches:",
      len(roundtrip_errors_52))

print("Validation-speaker overlap:", 0)
print("Quarantined examples included:", 0)

print("Primary validation examples:",
      len(selection_52["primary_val_ids"]))

# ------------------------------------------------------------
# 8. Randomly spot-check newly selected recordings
# ------------------------------------------------------------

new_records_52 = [
    r for r in selected_records_52
    if r["id"] not in set(carryover_52)
]

sample_rng_52 = random.Random(SEED_52 + 1)
spot_checks_52 = sample_rng_52.sample(
    new_records_52,
    min(4, len(new_records_52))
)

print("\nRANDOM AUDIO QUALITY CHECK")
print("=" * 60)

for i, record in enumerate(spot_checks_52, 1):

    print(f"\nSAMPLE {i}")
    print("ID:", record["id"])
    print("Duration:", record["duration_seconds"])
    print("Transcript:", record["transcription"])

    display(Audio(
        filename=str(
            Path(AUDIO_DIR) / record["audio_file"]
        )
    ))

print("\nSELECTION COMPLETE")
print("Saved:", SELECTION_52)
print("No training or model checkpoint modifications.")


## Original source cell 57

Position 58 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 53 - TWO-HOUR HEAD-ONLY ASR TRAINING
#
# 452 training recordings / 15 development recordings
# Frozen encoder: best model from Cell 50
# Trainable: CTC head only (329,025 parameters)
#
# 5 epochs, full FP32
# Resumable checkpoints after every epoch
# All earlier checkpoints remain unchanged
# ============================================================

import gc
import copy
import json
import math
import random
import hashlib
from pathlib import Path

import torch
import torch.nn.functional as F
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

DEVICE_53 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ROOT_53 = DATA_DIR / "amharic_2hour_pilot_v1"

SOURCE_53 = (
    DATA_DIR / "scaled_amharic_64_16_v1"
    / "stage2_filtered_62_v1" / "best.pt"
)

SELECTION_53 = ROOT_53 / "selection.json"

RUN_53 = ROOT_53 / "head_only_stage1_v1"
RUN_53.mkdir(parents=True, exist_ok=True)

LATEST_53 = RUN_53 / "latest.pt"
BEST_53 = RUN_53 / "best.pt"
HISTORY_53 = RUN_53 / "history.json"

EPOCHS_53 = 5
HEAD_LR_53 = 5e-4
CLIP_53 = 5.0
SEED_53 = 20261020
BLANK_53 = tokenizer.pad_token_id

assert SOURCE_53.is_file()
assert SELECTION_53.is_file()

selection_53 = json.loads(
    SELECTION_53.read_text(encoding="utf-8")
)

train_ids_53 = selection_53["train_ids"]
val_ids_53 = selection_53["primary_val_ids"]
carryover_53 = selection_53["carryover_ids"]

assert len(train_ids_53) == 452
assert len(val_ids_53) == 15
assert len(set(train_ids_53 + val_ids_53)) == 467

records_53 = {r["id"]: r for r in train_records}
indices_53 = {
    r["id"]: i
    for i, r in enumerate(train_dataset.records)
}

assert all(
    rid in indices_53
    for rid in train_ids_53 + val_ids_53
)

train_speakers_53 = {
    records_53[rid]["speaker_id"]
    for rid in train_ids_53
}

val_speakers_53 = {
    records_53[rid]["speaker_id"]
    for rid in val_ids_53
}

assert not train_speakers_53.intersection(val_speakers_53)

new_ids_53 = [
    rid for rid in train_ids_53
    if rid not in set(carryover_53)
]

# Fixed training probe, separate from evaluation.
rng_probe_53 = random.Random(SEED_53)
new_probe_53 = rng_probe_53.sample(
    new_ids_53, min(64, len(new_ids_53))
)

print("TWO-HOUR HEAD-ONLY EXPERIMENT")
print("=" * 60)
print("Training examples:", len(train_ids_53))
print("Development examples:", len(val_ids_53))
print("Previously used training examples:", len(carryover_53))
print("New training examples:", len(new_ids_53))
print("Distinct training speakers:", len(train_speakers_53))
print("Train/development speaker overlap:", 0)
print("Device:", DEVICE_53)

# ------------------------------------------------------------
# 1. Load adapted encoder and trained head
# ------------------------------------------------------------

source_53 = torch.load(
    SOURCE_53,
    map_location="cpu",
    weights_only=True
)

assert source_53["precision"] == "fp32"
assert len(source_53["upper_layers"]) == 2

model_53 = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_53).float()

model_53.lm_head.load_state_dict(
    source_53["head"], strict=True
)

for layer, weights in zip(
    model_53.wav2vec2.encoder.layers[-2:],
    source_53["upper_layers"]
):
    layer.load_state_dict(weights, strict=True)

model_53.eval()
model_53.requires_grad_(False)

# Copy the trainable head outside inference_mode.
head_53 = copy.deepcopy(
    model_53.lm_head
).to(DEVICE_53).float()

head_53.requires_grad_(True)

assert sum(
    p.numel() for p in head_53.parameters()
) == 329025

print("\nStarting model restored.")
print("Encoder: frozen")
print("CTC head: trainable")
print("Head parameters: 329,025")

# ------------------------------------------------------------
# 2. Cache frozen encoder features in CPU RAM
# ------------------------------------------------------------

# Each item stores its encoder output as float32.
# This avoids running the 300M-parameter XLS-R encoder
# again on every training epoch.

def cache_record_53(rid):

    example = train_dataset[indices_53[rid]]
    batch = ctc_collator([example])

    input_values = batch["input_values"].to(
        DEVICE_53
    )

    attention_mask = batch["attention_mask"].to(
        DEVICE_53
    )

    with torch.no_grad():
        hidden = model_53.wav2vec2(
            input_values=input_values,
            attention_mask=attention_mask
        ).last_hidden_state

    features = hidden.detach().cpu().float().contiguous()

    targets = batch["labels"][0]
    targets = targets[
        targets >= 0
    ].detach().cpu().long().contiguous()

    repeats = int(
        (targets[1:] == targets[:-1]).sum().item()
    )

    assert features.shape[1] >= (
        targets.numel() + repeats
    )

    assert not (targets == BLANK_53).any()
    assert torch.isfinite(features).all().item()

    reference = normalize_amharic_ctc(
        records_53[rid]["transcription"]
    )

    return {
        "features": features,
        "targets": targets,
        "frames": int(features.shape[1]),
        "target_length": int(targets.numel()),
        "reference": reference
    }

print("\nCaching frozen encoder features...")
print("Features are stored in CPU RAM.")

cache_53 = {}

all_ids_53 = train_ids_53 + val_ids_53

for i, rid in enumerate(all_ids_53, 1):

    cache_53[rid] = cache_record_53(rid)

    if i % 50 == 0 or i == len(all_ids_53):
        print(f"Cached: {i}/{len(all_ids_53)}")

cached_bytes_53 = sum(
    item["features"].numel() * 4
    for item in cache_53.values()
)

print(
    "Cached feature memory:",
    round(cached_bytes_53 / 1024**3, 2),
    "GiB"
)

# The encoder is no longer needed for head training.
del model_53
gc.collect()

if DEVICE_53.type == "cuda":
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Setup optimizer and reproducible experiment ID
# ------------------------------------------------------------

optimizer_53 = torch.optim.AdamW(
    head_53.parameters(),
    lr=HEAD_LR_53,
    weight_decay=0.0
)

config_53 = {
    "source_checkpoint": str(SOURCE_53),
    "source_update": source_53.get("updates"),
    "training_ids": train_ids_53,
    "development_ids": val_ids_53,
    "references": {
        rid: records_53[rid]["transcription"]
        for rid in all_ids_53
    },
    "head_lr": HEAD_LR_53,
    "clip_norm": CLIP_53,
    "seed": SEED_53,
    "epochs": EPOCHS_53
}

signature_53 = hashlib.sha256(
    json.dumps(
        config_53,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

# ------------------------------------------------------------
# 4. Resume existing experiment if available
# ------------------------------------------------------------

start_epoch_53 = 0
history_53 = []
best_cer_53 = float("inf")

if LATEST_53.exists():

    previous_53 = torch.load(
        LATEST_53,
        map_location="cpu",
        weights_only=True
    )

    assert previous_53["signature"] == signature_53

    head_53.load_state_dict(
        previous_53["head"]
    )

    optimizer_53.load_state_dict(
        previous_53["optimizer"]
    )

    start_epoch_53 = int(
        previous_53["completed_epochs"]
    )

    best_cer_53 = float(
        previous_53["best_cer"]
    )

    if HISTORY_53.exists():
        history_53 = json.loads(
            HISTORY_53.read_text(encoding="utf-8")
        )

    print(
        "\nResuming from completed epoch:",
        start_epoch_53
    )

else:
    print("\nStarting new head-only training.")

assert 0 <= start_epoch_53 <= EPOCHS_53

# ------------------------------------------------------------
# 5. CTC training loss
# ------------------------------------------------------------

def loss_53(rid):

    item = cache_53[rid]

    features = item["features"].to(DEVICE_53)
    targets = item["targets"].to(DEVICE_53)

    logits = head_53(features)

    log_probs = F.log_softmax(
        logits.float(), dim=-1
    ).transpose(0, 1)

    input_lengths = torch.tensor(
        [item["frames"]],
        dtype=torch.long
    )

    target_lengths = torch.tensor(
        [item["target_length"]],
        dtype=torch.long
    )

    return F.ctc_loss(
        log_probs,
        targets,
        input_lengths,
        target_lengths,
        blank=BLANK_53,
        reduction="mean",
        zero_infinity=False
    )

# ------------------------------------------------------------
# 6. Evaluation
# ------------------------------------------------------------

def evaluate_53(ids):

    head_53.eval()

    references = []
    predictions = []

    blank_count = 0
    frame_count = 0

    with torch.no_grad():

        for rid in ids:

            item = cache_53[rid]

            features = item["features"].to(
                DEVICE_53
            )

            logits = head_53(features)

            predicted_ids = logits.argmax(
                dim=-1
            )

            blank_count += int(
                (predicted_ids == BLANK_53).sum().item()
            )

            frame_count += predicted_ids.numel()

            prediction = processor.batch_decode(
                predicted_ids.cpu()
            )[0]

            prediction = normalize_amharic_ctc(
                prediction
            )

            references.append(item["reference"])
            predictions.append(prediction)

    reference_chars = sum(
        len(s.replace(" ", ""))
        for s in references
    )

    predicted_chars = sum(
        len(s.replace(" ", ""))
        for s in predictions
    )

    return {
        "cer": float(cer(references, predictions)),
        "wer": float(wer(references, predictions)),
        "length_ratio": (
            predicted_chars / max(reference_chars, 1)
        ),
        "blank_fraction": (
            blank_count / max(frame_count, 1)
        ),
        "nonempty": sum(
            bool(s.strip()) for s in predictions
        ),
        "example": predictions[0]
    }

def report_53(epoch):

    old = evaluate_53(carryover_53)
    new = evaluate_53(new_probe_53)
    dev = evaluate_53(val_ids_53)

    result = {
        "epoch": epoch,
        "updates": epoch * len(train_ids_53),
        "old_train_cer": old["cer"],
        "new_train_probe_cer": new["cer"],
        "dev_cer": dev["cer"],
        "dev_wer": dev["wer"],
        "dev_length_ratio": dev["length_ratio"],
        "dev_blank_fraction": dev["blank_fraction"],
        "dev_nonempty": dev["nonempty"],
        "dev_example": dev["example"]
    }

    print(f"\nEPOCH {epoch}/{EPOCHS_53}")
    print("-" * 55)
    print(
        "Previously trained CER:",
        f"{old['cer']*100:.2f}%"
    )
    print(
        "New training probe CER:",
        f"{new['cer']*100:.2f}%"
    )
    print(
        "Development CER:",
        f"{dev['cer']*100:.2f}%"
    )
    print(
        "Development WER:",
        f"{dev['wer']*100:.2f}%"
    )
    print(
        "Prediction/reference length:",
        f"{dev['length_ratio']:.2f}"
    )
    print(
        "Blank frames:",
        f"{dev['blank_fraction']*100:.2f}%"
    )
    print(
        "Nonempty development predictions:",
        f"{dev['nonempty']}/15"
    )
    print(
        "Development example:",
        dev["example"][:120] or "[EMPTY]"
    )

    return result

# ------------------------------------------------------------
# 7. Checkpoint management
# ------------------------------------------------------------

def save_53(epoch, improved):

    state = {
        "signature": signature_53,
        "completed_epochs": epoch,
        "best_cer": best_cer_53,
        "source_checkpoint": str(SOURCE_53),
        "head": head_53.state_dict(),
        "optimizer": optimizer_53.state_dict()
    }

    temporary = RUN_53 / "latest.pt.partial"
    torch.save(state, temporary)
    temporary.replace(LATEST_53)

    if improved:
        temporary_best = RUN_53 / "best.pt.partial"
        torch.save(state, temporary_best)
        temporary_best.replace(BEST_53)

    temporary_history = RUN_53 / "history.json.partial"

    temporary_history.write_text(
        json.dumps(
            history_53,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    temporary_history.replace(HISTORY_53)

# ------------------------------------------------------------
# 8. Establish baseline
# ------------------------------------------------------------

print("\nTWO-HOUR FROZEN-ENCODER TRAINING")
print("=" * 60)
print("Epochs:", EPOCHS_53)
print("Total planned optimizer updates:",
      EPOCHS_53 * len(train_ids_53))
print("Precision: FP32")
print("Head LR:", HEAD_LR_53)
print("Gradient clipping:", CLIP_53)

if start_epoch_53 == 0 and not history_53:

    initial_53 = report_53(0)
    history_53.append(initial_53)
    best_cer_53 = initial_53["dev_cer"]

# ------------------------------------------------------------
# 9. Train CTC head for five epochs
# ------------------------------------------------------------

for epoch in range(
    start_epoch_53 + 1,
    EPOCHS_53 + 1
):

    order = list(train_ids_53)

    random.Random(
        SEED_53 + epoch
    ).shuffle(order)

    losses = []
    gradient_norms = []

    head_53.train()

    for i, rid in enumerate(order, 1):

        optimizer_53.zero_grad(set_to_none=True)

        loss = loss_53(rid)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Invalid CTC loss: {rid}"
            )

        loss.backward()

        norm = torch.nn.utils.clip_grad_norm_(
            head_53.parameters(),
            max_norm=CLIP_53,
            error_if_nonfinite=True
        )

        optimizer_53.step()

        losses.append(float(loss.item()))
        gradient_norms.append(float(norm))

        if i % 100 == 0 or i == len(order):
            print(
                f"Epoch {epoch}: "
                f"{i}/{len(order)} updates"
            )

    result = report_53(epoch)

    result["mean_training_loss"] = (
        sum(losses) / len(losses)
    )

    result["median_gradient_norm"] = (
        sorted(gradient_norms)[
            len(gradient_norms) // 2
        ]
    )

    result["clipped_fraction"] = (
        sum(n > CLIP_53 for n in gradient_norms)
        / len(gradient_norms)
    )

    improved = result["dev_cer"] < best_cer_53

    if improved:
        best_cer_53 = result["dev_cer"]

    history_53.append(result)
    save_53(epoch, improved)

    print(
        "Mean CTC training loss:",
        round(result["mean_training_loss"], 4)
    )

    print(
        "Median gradient norm:",
        round(result["median_gradient_norm"], 3)
    )

    print(
        "Gradient clipping frequency:",
        f"{result['clipped_fraction']*100:.1f}%"
    )

    # Stop early if severe blank collapse develops.
    if (
        result["dev_length_ratio"] < 0.10
        and result["dev_blank_fraction"] > 0.98
    ):
        print(
            "\nEARLY STOP: Severe under-generation "
            "and blank dominance detected."
        )
        break

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("TWO-HOUR HEAD-ONLY TRAINING RESULTS")
print("=" * 75)

print(
    f"{'Epoch':<8}"
    f"{'Old CER':<13}"
    f"{'New CER':<13}"
    f"{'Dev CER':<13}"
    f"{'Dev WER':<13}"
    f"{'Length':<10}"
)

for row in history_53:

    print(
        f"{row['epoch']:<8}"
        f"{row['old_train_cer']*100:<13.2f}"
        f"{row['new_train_probe_cer']*100:<13.2f}"
        f"{row['dev_cer']*100:<13.2f}"
        f"{row['dev_wer']*100:<13.2f}"
        f"{row['dev_length_ratio']:<10.2f}"
    )

print(
    "\nBest development CER:",
    f"{best_cer_53*100:.2f}%"
)

print("Latest checkpoint:", LATEST_53)

if BEST_53.exists():
    print("Best improved checkpoint:", BEST_53)
else:
    print(
        "No improvement over starting model. "
        "Use the original source checkpoint."
    )

print("History:", HISTORY_53)
print("All previous checkpoints unchanged.")


## Original source cell 58

Position 59 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 54 - CTC BLANK-BIAS DIAGNOSTIC
#
# Compare warm start, epoch 1, and epoch 4.
# Test blank-logit penalties WITHOUT training.
#
# Uses cached XLS-R features from Cell 53.
# No model checkpoints modified.
# ============================================================

import json
import random
from pathlib import Path

import torch
from jiwer import cer, wer

ROOT_54 = DATA_DIR / "amharic_2hour_pilot_v1"
RUN_54 = ROOT_54 / "head_only_stage1_v1"

SOURCE_54 = (
    DATA_DIR / "scaled_amharic_64_16_v1"
    / "stage2_filtered_62_v1" / "best.pt"
)

CHECKPOINTS_54 = {
    "Warm start": SOURCE_54,
    "Epoch 1": RUN_54 / "best.pt",
    "Epoch 4": RUN_54 / "latest.pt"
}

assert "cache_53" in globals(), (
    "Cell 53's cached encoder features are unavailable. "
    "Do not rerun training just to restore them."
)

selection_54 = json.loads(
    (ROOT_54 / "selection.json").read_text(
        encoding="utf-8"
    )
)

train_ids_54 = selection_54["train_ids"]
val_ids_54 = selection_54["primary_val_ids"]
carryover_54 = set(selection_54["carryover_ids"])

new_train_ids_54 = [
    rid for rid in train_ids_54
    if rid not in carryover_54
]

train_probe_54 = random.Random(20261020).sample(
    new_train_ids_54, 64
)

all_ids_54 = train_probe_54 + val_ids_54

assert len(train_probe_54) == 64
assert len(val_ids_54) == 15
assert all(rid in cache_53 for rid in all_ids_54)

blank_id_54 = tokenizer.pad_token_id
penalties_54 = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]

device_54 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("CTC BLANK-BIAS DIAGNOSTIC")
print("=" * 60)
print("Training probe:", len(train_probe_54))
print("Development recordings:", len(val_ids_54))
print("Penalties:", penalties_54)
print("No training will be performed.")

# ------------------------------------------------------------
# 1. Decode cached logits with an optional blank penalty
# ------------------------------------------------------------

def decode_logits_54(logits, penalty):

    adjusted = logits.clone()

    adjusted[:, :, blank_id_54] -= penalty

    token_ids = adjusted.argmax(dim=-1)

    text = processor.batch_decode(
        token_ids
    )[0]

    return (
        normalize_amharic_ctc(text),
        int((token_ids == blank_id_54).sum().item()),
        token_ids.numel()
    )

def evaluate_54(logit_cache, record_ids, penalty):

    references = []
    predictions = []

    blank_frames = 0
    all_frames = 0

    for rid in record_ids:

        prediction, blanks, frames = decode_logits_54(
            logit_cache[rid], penalty
        )

        references.append(
            cache_53[rid]["reference"]
        )

        predictions.append(prediction)

        blank_frames += blanks
        all_frames += frames

    ref_characters = sum(
        len(s.replace(" ", ""))
        for s in references
    )

    predicted_characters = sum(
        len(s.replace(" ", ""))
        for s in predictions
    )

    return {
        "cer": float(cer(references, predictions)),
        "wer": float(wer(references, predictions)),
        "length_ratio": (
            predicted_characters / max(ref_characters, 1)
        ),
        "blank_fraction": (
            blank_frames / max(all_frames, 1)
        ),
        "nonempty": sum(bool(s.strip()) for s in predictions),
        "sample_prediction": predictions[0]
    }

# ------------------------------------------------------------
# 2. Compare all three heads using identical features
# ------------------------------------------------------------

all_results_54 = {}

expected_dev_54 = {
    "Warm start": 0.8907,
    "Epoch 1": 0.8882,
    "Epoch 4": 0.9592
}

for model_name, path in CHECKPOINTS_54.items():

    assert path.is_file(), f"Missing checkpoint: {path}"

    saved = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    head_weights = saved["head"]

    weight = head_weights["weight"]

    diagnostic_head = torch.nn.Linear(
        in_features=weight.shape[1],
        out_features=weight.shape[0],
        bias=True
    ).to(device_54).float()

    diagnostic_head.load_state_dict(
        head_weights,
        strict=True
    )

    diagnostic_head.eval()

    # Compute acoustic logits only once per recording.
    # Keep logits in CPU RAM for the penalty sweep.
    logit_cache_54 = {}

    with torch.no_grad():

        for rid in all_ids_54:

            features = cache_53[rid]["features"].to(
                device_54
            )

            logits = diagnostic_head(features)

            logit_cache_54[rid] = (
                logits.detach().cpu().float()
            )

    model_results = []

    for penalty in penalties_54:

        train_result = evaluate_54(
            logit_cache_54,
            train_probe_54,
            penalty
        )

        dev_result = evaluate_54(
            logit_cache_54,
            val_ids_54,
            penalty
        )

        model_results.append({
            "penalty": penalty,
            "train_cer": train_result["cer"],
            "dev_cer": dev_result["cer"],
            "dev_wer": dev_result["wer"],
            "dev_length_ratio": dev_result["length_ratio"],
            "dev_blank_fraction": dev_result["blank_fraction"],
            "dev_nonempty": dev_result["nonempty"],
            "dev_sample": dev_result["sample_prediction"]
        })

    # Verify our zero-penalty baseline matches Cell 53.
    actual_baseline = model_results[0]["dev_cer"]

    assert abs(
        actual_baseline - expected_dev_54[model_name]
    ) < 0.005, (
        f"Baseline mismatch for {model_name}: "
        f"{actual_baseline:.4f}"
    )

    all_results_54[model_name] = model_results

    print("\n" + "=" * 72)
    print(model_name.upper())
    print("=" * 72)

    print(
        f"{'Penalty':<10}"
        f"{'Train CER':<15}"
        f"{'Dev CER':<14}"
        f"{'Dev length':<15}"
        f"{'Dev blank':<14}"
    )

    for row in model_results:

        print(
            f"{row['penalty']:<10.1f}"
            f"{row['train_cer']*100:<15.2f}"
            f"{row['dev_cer']*100:<14.2f}"
            f"{row['dev_length_ratio']:<15.2f}"
            f"{row['dev_blank_fraction']*100:<14.2f}"
        )

    # Pick an adjustment using training CER, not
    # development CER, to reduce validation tuning.
    chosen = min(
        model_results,
        key=lambda r: (
            r["train_cer"],
            r["penalty"]
        )
    )

    print(
        "\nBest training-probe penalty:",
        chosen["penalty"]
    )

    print(
        "Development CER at chosen penalty:",
        f"{chosen['dev_cer']*100:.2f}%"
    )

    print(
        "Development WER:",
        f"{chosen['dev_wer']*100:.2f}%"
    )

    print(
        "Development prediction/reference length:",
        f"{chosen['dev_length_ratio']:.2f}"
    )

    print(
        "Example prediction:",
        chosen["dev_sample"][:150] or "[EMPTY]"
    )

    del diagnostic_head, logit_cache_54

# ------------------------------------------------------------
# 3. Save diagnostic results
# ------------------------------------------------------------

output_54 = RUN_54 / "blank_bias_diagnostic.json"

output_54.write_text(
    json.dumps(
        {
            "description": (
                "Exploratory blank-logit penalty diagnostic. "
                "Penalties are not learned model parameters."
            ),
            "train_probe_count": len(train_probe_54),
            "development_count": len(val_ids_54),
            "results": all_results_54
        },
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 60)
print("BLANK-BIAS DIAGNOSTIC FINISHED")
print("=" * 60)
print("Results saved:", output_54)
print("Optimizer updates: 0")
print("Model checkpoints modified: 0")


## Original source cell 59

Position 60 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 55 - CTC PREFIX BEAM SEARCH DIAGNOSTIC
#
# Compare greedy and beam decoding for:
#   Epoch 1: blank penalty 2.0
#   Epoch 4: blank penalty 3.0
#
# 24 new training examples + 15 development examples.
# No gradients, optimizer steps, or checkpoint changes.
# ============================================================

import json
import math
import random
from pathlib import Path

import torch
from jiwer import cer, wer

ROOT_55 = DATA_DIR / "amharic_2hour_pilot_v1"
RUN_55 = ROOT_55 / "head_only_stage1_v1"

SOURCE_55 = (
    DATA_DIR / "scaled_amharic_64_16_v1"
    / "stage2_filtered_62_v1" / "best.pt"
)

selection_55 = json.loads(
    (ROOT_55 / "selection.json").read_text(
        encoding="utf-8"
    )
)

carryover_55 = set(selection_55["carryover_ids"])

new_train_55 = [
    rid for rid in selection_55["train_ids"]
    if rid not in carryover_55
]

train_probe_55 = random.Random(551).sample(
    new_train_55, 24
)

dev_ids_55 = selection_55["primary_val_ids"]
all_ids_55 = train_probe_55 + dev_ids_55

assert len(train_probe_55) == 24
assert len(dev_ids_55) == 15

blank_55 = tokenizer.pad_token_id
device_55 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

heads_55 = {
    "Epoch 1": {
        "path": RUN_55 / "best.pt",
        "penalty": 2.0,
        "expected_greedy_cer": 0.8882
    },
    "Epoch 4": {
        "path": RUN_55 / "latest.pt",
        "penalty": 3.0,
        "expected_greedy_cer": 0.9592
    }
}

# ------------------------------------------------------------
# 1. Access cached encoder features.
#    Reconstruct only the required features if needed.
# ------------------------------------------------------------

available_cache_55 = globals().get("cache_53", {})

features_55 = {}

if all(rid in available_cache_55 for rid in all_ids_55):

    print("Reusing Cell 53's frozen encoder features.")

    for rid in all_ids_55:
        features_55[rid] = {
            "features": available_cache_55[rid][
                "features"
            ],
            "reference": available_cache_55[rid][
                "reference"
            ]
        }

else:

    print("Reconstructing the required frozen features.")
    print("No training will occur.")

    from transformers import Wav2Vec2ForCTC

    source = torch.load(
        SOURCE_55,
        map_location="cpu",
        weights_only=True
    )

    encoder_model = Wav2Vec2ForCTC.from_pretrained(
        str(BASELINE_DIR)
    ).to(device_55).float()

    for layer, weights in zip(
        encoder_model.wav2vec2.encoder.layers[-2:],
        source["upper_layers"]
    ):
        layer.load_state_dict(weights, strict=True)

    encoder_model.eval()
    encoder_model.requires_grad_(False)

    records_55 = {
        r["id"]: r for r in train_records
    }

    indices_55 = {
        r["id"]: i
        for i, r in enumerate(train_dataset.records)
    }

    with torch.no_grad():

        for i, rid in enumerate(all_ids_55, 1):

            example = train_dataset[
                indices_55[rid]
            ]

            batch = ctc_collator([example])

            hidden = encoder_model.wav2vec2(
                input_values=batch[
                    "input_values"
                ].to(device_55),
                attention_mask=batch[
                    "attention_mask"
                ].to(device_55)
            ).last_hidden_state

            features_55[rid] = {
                "features": hidden.detach().cpu().float(),
                "reference": normalize_amharic_ctc(
                    records_55[rid]["transcription"]
                )
            }

            if i % 10 == 0 or i == len(all_ids_55):
                print("Features:", i, "/", len(all_ids_55))

    del encoder_model

    if device_55.type == "cuda":
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 2. Stable log-space addition
# ------------------------------------------------------------

NEG_55 = -float("inf")

def logadd_55(a, b):
    if a == NEG_55:
        return b
    if b == NEG_55:
        return a

    maximum = max(a, b)

    return maximum + math.log1p(
        math.exp(min(a, b) - maximum)
    )

# ------------------------------------------------------------
# 3. Actual CTC prefix beam search
#
# Each prefix stores:
# pb  = probability ending in blank
# pnb = probability ending in nonblank
#
# Repeated tokens are handled using CTC rules.
# ------------------------------------------------------------

def prefix_beam_55(
    logits,
    blank_penalty=0.0,
    beam_width=10,
    top_k=12
):

    scores = logits.squeeze(0).float().clone()
    scores[:, blank_55] -= blank_penalty

    log_probs = torch.log_softmax(
        scores, dim=-1
    )

    blank_scores = log_probs[
        :, blank_55
    ].tolist()

    nonblank_scores = log_probs.clone()
    nonblank_scores[:, blank_55] = NEG_55

    values, indices = torch.topk(
        nonblank_scores,
        k=min(top_k, nonblank_scores.shape[-1] - 1),
        dim=-1
    )

    values = values.tolist()
    indices = indices.tolist()

    # Empty prefix initially ends in blank.
    beams = {(): (0.0, NEG_55)}

    for t in range(log_probs.shape[0]):

        next_beams = {}
        blank_score = blank_scores[t]

        def update(prefix, index, value):

            if value == NEG_55:
                return

            if prefix not in next_beams:
                next_beams[prefix] = [
                    NEG_55, NEG_55
                ]

            old = next_beams[prefix][index]

            next_beams[prefix][index] = (
                logadd_55(old, value)
            )

        for prefix, (pb, pnb) in beams.items():

            total = logadd_55(pb, pnb)

            # Emitting blank preserves the prefix.
            update(
                prefix, 0,
                total + blank_score
            )

            for token, score in zip(
                indices[t], values[t]
            ):

                if prefix and token == prefix[-1]:

                    # Repeating a token without an
                    # intervening blank collapses it.
                    update(
                        prefix, 1,
                        pnb + score
                    )

                    # The same token following blank
                    # creates a new repeated character.
                    update(
                        prefix + (token,), 1,
                        pb + score
                    )

                else:

                    update(
                        prefix + (token,), 1,
                        total + score
                    )

        best = sorted(
            next_beams.items(),
            key=lambda item: logadd_55(
                item[1][0], item[1][1]
            ),
            reverse=True
        )[:beam_width]

        beams = {
            prefix: tuple(probabilities)
            for prefix, probabilities in best
        }

    best_prefix = max(
        beams.items(),
        key=lambda item: logadd_55(
            item[1][0], item[1][1]
        )
    )[0]

    # Beam output is ALREADY CTC-collapsed.
    # Do not collapse repeated symbols again.
    decoded = tokenizer.decode(
        list(best_prefix),
        group_tokens=False
    )

    return normalize_amharic_ctc(decoded)

# ------------------------------------------------------------
# 4. Greedy decoding for direct comparison
# ------------------------------------------------------------

def greedy_55(logits, blank_penalty=0.0):

    adjusted = logits.clone().float()
    adjusted[:, :, blank_55] -= blank_penalty

    token_ids = adjusted.argmax(dim=-1)

    return normalize_amharic_ctc(
        processor.batch_decode(token_ids)[0]
    )

# ------------------------------------------------------------
# 5. Calculate CER, WER, and output length
# ------------------------------------------------------------

def measure_55(predictions, ids):

    references = [
        features_55[rid]["reference"]
        for rid in ids
    ]

    hypotheses = [
        predictions[rid]
        for rid in ids
    ]

    ref_characters = sum(
        len(s.replace(" ", ""))
        for s in references
    )

    hyp_characters = sum(
        len(s.replace(" ", ""))
        for s in hypotheses
    )

    return {
        "cer": float(cer(references, hypotheses)),
        "wer": float(wer(references, hypotheses)),
        "length_ratio": (
            hyp_characters / max(ref_characters, 1)
        ),
        "example": hypotheses[0]
    }

# ------------------------------------------------------------
# 6. Run both decoders with both settings
# ------------------------------------------------------------

results_55 = {}

print("\nCTC BEAM SEARCH COMPARISON")
print("=" * 65)
print("Training probe: 24 recordings")
print("Development: 15 recordings")
print("Beam width: 10")
print("Nonblank candidates/frame: 12")

for name, configuration in heads_55.items():

    checkpoint = torch.load(
        configuration["path"],
        map_location="cpu",
        weights_only=True
    )

    weights = checkpoint["head"]
    penalty = configuration["penalty"]

    head = torch.nn.Linear(
        in_features=weights["weight"].shape[1],
        out_features=weights["weight"].shape[0],
        bias=True
    ).to(device_55).float()

    head.load_state_dict(weights, strict=True)
    head.eval()

    logits_cache = {}

    with torch.no_grad():

        for rid in all_ids_55:

            features = features_55[rid][
                "features"
            ].to(device_55)

            logits_cache[rid] = (
                head(features).detach().cpu().float()
            )

    strategies = [
        ("Greedy", "greedy", 0.0),
        ("Greedy + penalty", "greedy", penalty),
        ("Beam", "beam", 0.0),
        ("Beam + penalty", "beam", penalty)
    ]

    model_results = []

    print("\n" + "=" * 65)
    print(name.upper())
    print("=" * 65)

    for label, method, amount in strategies:

        predictions = {}

        for rid in all_ids_55:

            logits = logits_cache[rid]

            if method == "greedy":
                predictions[rid] = greedy_55(
                    logits, amount
                )
            else:
                predictions[rid] = prefix_beam_55(
                    logits,
                    blank_penalty=amount,
                    beam_width=10,
                    top_k=12
                )

        train = measure_55(
            predictions, train_probe_55
        )

        dev = measure_55(
            predictions, dev_ids_55
        )

        model_results.append({
            "strategy": label,
            "penalty": amount,
            "train_cer": train["cer"],
            "dev_cer": dev["cer"],
            "dev_wer": dev["wer"],
            "dev_length": dev["length_ratio"],
            "dev_example": dev["example"]
        })

        print(
            f"{label:<21}"
            f"train CER={train['cer']*100:6.2f}%  "
            f"dev CER={dev['cer']*100:6.2f}%  "
            f"dev WER={dev['wer']*100:6.2f}%  "
            f"length={dev['length_ratio']:.2f}"
        )

    baseline = model_results[0]["dev_cer"]

    assert abs(
        baseline -
        configuration["expected_greedy_cer"]
    ) < 0.005, (
        f"Unexpected baseline mismatch for {name}: "
        f"{baseline:.4f}"
    )

    # The blank penalty was selected in Cell 54
    # using its training probe, not this dev set.
    chosen = min(
        model_results,
        key=lambda row: row["train_cer"]
    )

    print("\nLowest training-probe CER:",
          chosen["strategy"])

    print(
        "Development CER:",
        f"{chosen['dev_cer']*100:.2f}%"
    )

    print(
        "Example transcription:",
        chosen["dev_example"][:180] or "[EMPTY]"
    )

    results_55[name] = model_results

    del head, logits_cache

# ------------------------------------------------------------
# 7. Save diagnostic results
# ------------------------------------------------------------

output_55 = RUN_55 / "ctc_beam_diagnostic.json"

output_55.write_text(
    json.dumps(
        {
            "beam_width": 10,
            "top_k_nonblank": 12,
            "train_probe_count": len(train_probe_55),
            "development_count": len(dev_ids_55),
            "description": (
                "Exploratory greedy and approximate CTC "
                "prefix beam decoding comparison."
            ),
            "results": results_55
        },
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("BEAM DIAGNOSTIC COMPLETE")
print("=" * 65)
print("Saved:", output_55)
print("Training updates: 0")
print("Checkpoints modified: 0")


## Original source cell 60

Position 61 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 56 - TWO-HOUR JOINT ENCODER + CTC HEAD PILOT
#
# Encoder: Cell 50 best checkpoint
# CTC head: Cell 53 epoch-1 checkpoint
#
# Train: 452 recordings, 408 speakers
# Development: 15 speaker-disjoint recordings
#
# Train last 2 encoder layers + CTC head
# One epoch, FP32, gradient clipping
# Checkpoint every 113 updates
#
# Previous checkpoints remain untouched.
# ============================================================

import gc
import json
import random
import hashlib
import statistics
from pathlib import Path

import torch
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

assert torch.cuda.is_available(), (
    "A CUDA GPU is required for this training experiment."
)

DEVICE_56 = torch.device("cuda")

ROOT_56 = DATA_DIR / "amharic_2hour_pilot_v1"

ENCODER_SOURCE_56 = (
    DATA_DIR / "scaled_amharic_64_16_v1"
    / "stage2_filtered_62_v1" / "best.pt"
)

HEAD_SOURCE_56 = (
    ROOT_56 / "head_only_stage1_v1" / "best.pt"
)

SELECTION_56 = ROOT_56 / "selection.json"

RUN_56 = ROOT_56 / "joint_last2_stage2_v1"
RUN_56.mkdir(parents=True, exist_ok=True)

LATEST_56 = RUN_56 / "latest.pt"
BEST_56 = RUN_56 / "best.pt"
HISTORY_56 = RUN_56 / "history.json"

ENCODER_LR_56 = 1e-5
HEAD_LR_56 = 1e-4
CLIP_NORM_56 = 5.0
BLANK_PENALTY_56 = 2.0

SEED_56 = 20261020
REPORT_EVERY_56 = 113
TOTAL_UPDATES_56 = 452

def read_json_56(path):
    assert path.is_file(), f"Missing file: {path}"
    return json.loads(path.read_text(encoding="utf-8"))

selection_56 = read_json_56(SELECTION_56)

train_ids_56 = selection_56["train_ids"]
val_ids_56 = selection_56["primary_val_ids"]
carryover_ids_56 = selection_56["carryover_ids"]

assert len(train_ids_56) == 452
assert len(val_ids_56) == 15
assert not set(train_ids_56) & set(val_ids_56)

records_56 = {
    r["id"]: r for r in train_records
}

indices_56 = {
    r["id"]: i
    for i, r in enumerate(train_dataset.records)
}

assert all(
    rid in indices_56
    for rid in train_ids_56 + val_ids_56
)

train_speakers_56 = {
    records_56[rid]["speaker_id"]
    for rid in train_ids_56
}

val_speakers_56 = {
    records_56[rid]["speaker_id"]
    for rid in val_ids_56
}

assert not train_speakers_56 & val_speakers_56

new_ids_56 = [
    rid for rid in train_ids_56
    if rid not in set(carryover_ids_56)
]

# Fixed diagnostic probes, not the full training set.
probe_rng_56 = random.Random(SEED_56)

old_probe_56 = probe_rng_56.sample(
    carryover_ids_56, 16
)

new_probe_56 = probe_rng_56.sample(
    new_ids_56, 32
)

# ------------------------------------------------------------
# 1. Restore the encoder and the epoch-1 head
# ------------------------------------------------------------

encoder_checkpoint_56 = torch.load(
    ENCODER_SOURCE_56,
    map_location="cpu",
    weights_only=True
)

head_checkpoint_56 = torch.load(
    HEAD_SOURCE_56,
    map_location="cpu",
    weights_only=True
)

assert encoder_checkpoint_56["precision"] == "fp32"
assert head_checkpoint_56["completed_epochs"] == 1

model_56 = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_56).float()

for layer, weights in zip(
    model_56.wav2vec2.encoder.layers[-2:],
    encoder_checkpoint_56["upper_layers"]
):
    layer.load_state_dict(weights, strict=True)

model_56.lm_head.load_state_dict(
    head_checkpoint_56["head"],
    strict=True
)

model_56.requires_grad_(False)

upper_layers_56 = list(
    model_56.wav2vec2.encoder.layers[-2:]
)

for layer in upper_layers_56:
    layer.requires_grad_(True)

model_56.lm_head.requires_grad_(True)

# Keep dropout and SpecAugment disabled for this
# controlled continuation. Gradients remain enabled.
model_56.eval()

encoder_params_56 = [
    p
    for layer in upper_layers_56
    for p in layer.parameters()
    if p.requires_grad
]

head_params_56 = list(
    model_56.lm_head.parameters()
)

trainable_56 = (
    encoder_params_56 + head_params_56
)

parameter_count_56 = sum(
    p.numel() for p in trainable_56
)

assert parameter_count_56 == 25521473

optimizer_56 = torch.optim.AdamW([
    {
        "params": encoder_params_56,
        "lr": ENCODER_LR_56,
        "weight_decay": 0.01
    },
    {
        "params": head_params_56,
        "lr": HEAD_LR_56,
        "weight_decay": 0.0
    }
])

config_56 = {
    "encoder_checkpoint": str(ENCODER_SOURCE_56),
    "head_checkpoint": str(HEAD_SOURCE_56),
    "train_ids": train_ids_56,
    "val_ids": val_ids_56,
    "encoder_lr": ENCODER_LR_56,
    "head_lr": HEAD_LR_56,
    "clip_norm": CLIP_NORM_56,
    "blank_penalty": BLANK_PENALTY_56,
    "seed": SEED_56,
    "total_updates": TOTAL_UPDATES_56
}

signature_56 = hashlib.sha256(
    json.dumps(
        config_56,
        ensure_ascii=False,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

# ------------------------------------------------------------
# 2. Restore existing pilot progress if present
# ------------------------------------------------------------

completed_56 = 0
history_56 = []
best_adjusted_cer_56 = float("inf")

if LATEST_56.exists():

    checkpoint = torch.load(
        LATEST_56,
        map_location="cpu",
        weights_only=True
    )

    assert checkpoint["signature"] == signature_56

    model_56.lm_head.load_state_dict(
        checkpoint["head"]
    )

    for layer, weights in zip(
        upper_layers_56,
        checkpoint["upper_layers"]
    ):
        layer.load_state_dict(weights)

    optimizer_56.load_state_dict(
        checkpoint["optimizer"]
    )

    completed_56 = int(checkpoint["updates"])
    best_adjusted_cer_56 = float(
        checkpoint["best_adjusted_cer"]
    )

    assert HISTORY_56.is_file()
    history_56 = read_json_56(HISTORY_56)

    print("Resuming at update:", completed_56)

else:
    print("Starting new joint fine-tuning pilot.")

assert 0 <= completed_56 <= TOTAL_UPDATES_56

# ------------------------------------------------------------
# 3. Input and decoding helpers
# ------------------------------------------------------------

def batch_56(rid):

    item = train_dataset[indices_56[rid]]
    batch = ctc_collator([item])

    return {
        key: value.to(DEVICE_56)
        for key, value in batch.items()
    }

def decode_56(logits, penalty):

    adjusted = logits.float().clone()

    adjusted[:, :, tokenizer.pad_token_id] -= penalty

    token_ids = adjusted.argmax(dim=-1)

    text = processor.batch_decode(
        token_ids.cpu()
    )[0]

    return normalize_amharic_ctc(text)

def evaluate_56(ids):

    model_56.eval()

    references = []
    standard_predictions = []
    adjusted_predictions = []

    with torch.no_grad():

        for rid in ids:

            batch = batch_56(rid)

            logits = model_56(
                input_values=batch["input_values"],
                attention_mask=batch["attention_mask"]
            ).logits

            references.append(
                normalize_amharic_ctc(
                    records_56[rid]["transcription"]
                )
            )

            standard_predictions.append(
                decode_56(logits, penalty=0.0)
            )

            adjusted_predictions.append(
                decode_56(
                    logits,
                    penalty=BLANK_PENALTY_56
                )
            )

            del batch, logits

    reference_chars = sum(
        len(x.replace(" ", ""))
        for x in references
    )

    predicted_chars = sum(
        len(x.replace(" ", ""))
        for x in adjusted_predictions
    )

    return {
        "greedy_cer": float(
            cer(references, standard_predictions)
        ),
        "adjusted_cer": float(
            cer(references, adjusted_predictions)
        ),
        "adjusted_wer": float(
            wer(references, adjusted_predictions)
        ),
        "adjusted_length_ratio": (
            predicted_chars / max(reference_chars, 1)
        ),
        "sample": adjusted_predictions[0]
    }

def report_56(step):

    old = evaluate_56(old_probe_56)
    new = evaluate_56(new_probe_56)
    val = evaluate_56(val_ids_56)

    row = {
        "updates": step,
        "old_adjusted_cer": old["adjusted_cer"],
        "new_adjusted_cer": new["adjusted_cer"],
        "dev_greedy_cer": val["greedy_cer"],
        "dev_adjusted_cer": val["adjusted_cer"],
        "dev_adjusted_wer": val["adjusted_wer"],
        "dev_adjusted_length": val[
            "adjusted_length_ratio"
        ],
        "dev_example": val["sample"]
    }

    print(f"\nUPDATE {step}/{TOTAL_UPDATES_56}")
    print("-" * 60)

    print(
        "Old training probe adjusted CER:",
        f"{old['adjusted_cer']*100:.2f}%"
    )

    print(
        "New training probe adjusted CER:",
        f"{new['adjusted_cer']*100:.2f}%"
    )

    print(
        "Development greedy CER:",
        f"{val['greedy_cer']*100:.2f}%"
    )

    print(
        "Development adjusted CER:",
        f"{val['adjusted_cer']*100:.2f}%"
    )

    print(
        "Development adjusted WER:",
        f"{val['adjusted_wer']*100:.2f}%"
    )

    print(
        "Adjusted prediction/reference length:",
        f"{val['adjusted_length_ratio']:.2f}"
    )

    print(
        "Example:",
        val["sample"][:140] or "[EMPTY]"
    )

    return row

def save_56(step, improved):

    state = {
        "signature": signature_56,
        "updates": step,
        "best_adjusted_cer": best_adjusted_cer_56,
        "precision": "fp32",
        "head": model_56.lm_head.state_dict(),
        "upper_layers": [
            layer.state_dict()
            for layer in upper_layers_56
        ],
        "optimizer": optimizer_56.state_dict()
    }

    temp = RUN_56 / "latest.pt.partial"
    torch.save(state, temp)
    temp.replace(LATEST_56)

    if improved:
        temp_best = RUN_56 / "best.pt.partial"
        torch.save(state, temp_best)
        temp_best.replace(BEST_56)

    history_temp = RUN_56 / "history.json.partial"

    history_temp.write_text(
        json.dumps(
            history_56,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    history_temp.replace(HISTORY_56)

# ------------------------------------------------------------
# 4. Verify longest-recording FP32 gradients
# ------------------------------------------------------------

print("\nJOINT TRAINING CONFIGURATION")
print("=" * 60)

print("Training recordings:", len(train_ids_56))
print("Training speakers:", len(train_speakers_56))
print("Development recordings:", len(val_ids_56))
print("Trainable parameters:", parameter_count_56)
print("Encoder LR:", ENCODER_LR_56)
print("Head LR:", HEAD_LR_56)
print("Gradient clipping:", CLIP_NORM_56)
print("CTC loss reduction:", model_56.config.ctc_loss_reduction)
print("Blank penalty for evaluation:", BLANK_PENALTY_56)
print("Precision: FP32")
print("Completed updates:", completed_56)

if completed_56 == 0:

    longest_id_56 = max(
        train_ids_56,
        key=lambda rid: float(
            records_56[rid]["duration_seconds"]
        )
    )

    print("\nLONG-RECORDING GRADIENT PREFLIGHT")
    print("-" * 60)

    optimizer_56.zero_grad(set_to_none=True)

    longest_batch_56 = batch_56(longest_id_56)

    torch.cuda.reset_peak_memory_stats()

    with torch.autocast(
        device_type="cuda",
        enabled=False
    ):
        test_loss_56 = model_56(
            **longest_batch_56
        ).loss

    assert torch.isfinite(test_loss_56).item()

    test_loss_56.backward()

    gradient_norm_56 = (
        torch.nn.utils.clip_grad_norm_(
            trainable_56,
            max_norm=CLIP_NORM_56,
            error_if_nonfinite=True
        )
    )

    print("Recording ID:", longest_id_56)
    print(
        "Duration:",
        records_56[longest_id_56]["duration_seconds"],
        "seconds"
    )
    print("Loss:", float(test_loss_56))
    print("Gradient norm:", float(gradient_norm_56))
    print(
        "Peak allocated GPU memory:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2
        ),
        "GiB"
    )

    optimizer_56.zero_grad(set_to_none=True)
    del longest_batch_56, test_loss_56

    torch.cuda.empty_cache()

    print("Preflight passed. Optimizer updates: 0")

# ------------------------------------------------------------
# 5. Evaluate exact warm-start baseline
# ------------------------------------------------------------

if completed_56 == 0 and not history_56:

    baseline_56 = report_56(0)

    # Compare with Cell 55's epoch-1 results.
    assert abs(
        baseline_56["dev_greedy_cer"] - 0.8882
    ) < 0.015, (
        "Baseline greedy CER mismatch. "
        "Inspect restored checkpoints."
    )

    assert abs(
        baseline_56["dev_adjusted_cer"] - 0.8332
    ) < 0.015, (
        "Baseline adjusted CER mismatch. "
        "Inspect encoder/head pairing."
    )

    history_56.append(baseline_56)

    best_adjusted_cer_56 = baseline_56[
        "dev_adjusted_cer"
    ]

# ------------------------------------------------------------
# 6. Deterministic one-epoch training order
# ------------------------------------------------------------

order_56 = list(train_ids_56)

random.Random(
    SEED_56 + 1
).shuffle(order_56)

assert len(order_56) == TOTAL_UPDATES_56

segment_losses_56 = []
segment_norms_56 = []

# ------------------------------------------------------------
# 7. Train once on each recording
# ------------------------------------------------------------

for update in range(
    completed_56 + 1,
    TOTAL_UPDATES_56 + 1
):

    rid = order_56[update - 1]

    optimizer_56.zero_grad(set_to_none=True)
    model_56.eval()

    batch = batch_56(rid)

    with torch.autocast(
        device_type="cuda",
        enabled=False
    ):
        loss = model_56(**batch).loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(
            f"Nonfinite loss at update {update}, "
            f"recording {rid}"
        )

    loss.backward()

    gradient_norm = (
        torch.nn.utils.clip_grad_norm_(
            trainable_56,
            max_norm=CLIP_NORM_56,
            error_if_nonfinite=True
        )
    )

    raw_norm = float(gradient_norm)

    if raw_norm > 5000:
        optimizer_56.zero_grad(set_to_none=True)
        raise RuntimeError(
            f"Extreme gradient at update {update}: "
            f"{raw_norm:.1f}"
        )

    optimizer_56.step()

    segment_losses_56.append(float(loss.item()))
    segment_norms_56.append(raw_norm)

    del batch, loss

    if (
        update % REPORT_EVERY_56 == 0
        or update == TOTAL_UPDATES_56
    ):

        result = report_56(update)

        result["mean_step_loss"] = (
            statistics.mean(segment_losses_56)
        )

        result["median_gradient_norm"] = (
            statistics.median(segment_norms_56)
        )

        result["max_gradient_norm"] = (
            max(segment_norms_56)
        )

        result["clipped_fraction"] = (
            sum(
                norm > CLIP_NORM_56
                for norm in segment_norms_56
            ) / len(segment_norms_56)
        )

        improved = (
            result["dev_adjusted_cer"]
            < best_adjusted_cer_56
        )

        if improved:
            best_adjusted_cer_56 = result[
                "dev_adjusted_cer"
            ]

        history_56.append(result)

        save_56(
            step=update,
            improved=improved
        )

        print(
            "Mean training-step loss:",
            round(result["mean_step_loss"], 4)
        )

        print(
            "Median gradient norm:",
            round(result["median_gradient_norm"], 2)
        )

        print(
            "Maximum gradient norm:",
            round(result["max_gradient_norm"], 2)
        )

        print(
            "Fraction clipped:",
            f"{result['clipped_fraction']*100:.1f}%"
        )

        segment_losses_56.clear()
        segment_norms_56.clear()

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("TWO-HOUR JOINT FINE-TUNING RESULTS")
print("=" * 78)

print(
    f"{'Update':<10}"
    f"{'Old CER':<13}"
    f"{'New CER':<13}"
    f"{'Dev greedy':<14}"
    f"{'Dev adjusted':<15}"
    f"{'Length':<9}"
)

for row in history_56:

    print(
        f"{row['updates']:<10}"
        f"{row['old_adjusted_cer']*100:<13.2f}"
        f"{row['new_adjusted_cer']*100:<13.2f}"
        f"{row['dev_greedy_cer']*100:<14.2f}"
        f"{row['dev_adjusted_cer']*100:<15.2f}"
        f"{row['dev_adjusted_length']:<9.2f}"
    )

print(
    "\nBest adjusted development CER:",
    f"{best_adjusted_cer_56*100:.2f}%"
)

print("Latest checkpoint:", LATEST_56)

if BEST_56.exists():
    print("Best joint checkpoint:", BEST_56)
else:
    print(
        "No joint checkpoint improved on the "
        "original epoch-1 model."
    )

print("History:", HISTORY_56)
print("Previous checkpoints unchanged.")


## Original source cell 61

Position 62 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 57 - CONTROLLED FOUR-RECORDING LEARNABILITY TEST
#
# Compare:
# A. Original pretrained XLS-R encoder
# B. Adapted encoder from Cell 50
#
# Both receive IDENTICALLY initialized fresh CTC heads.
# Train only on four manually reviewed recordings.
#
# 1600 updates per branch, save every 400.
# No development or test recordings used.
# ============================================================

import copy
import gc
import json
import random
import hashlib
import statistics
from pathlib import Path

import torch
import torch.nn.functional as F
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

assert torch.cuda.is_available(), "CUDA is required."

DEVICE_57 = torch.device("cuda")
ROOT_57 = DATA_DIR / "amharic_2hour_pilot_v1"
RUN_57 = ROOT_57 / "four_recording_learnability_v1"
RUN_57.mkdir(parents=True, exist_ok=True)

REVIEW_ROOT_57 = DATA_DIR / "scaled_amharic_64_16_v1"

ENCODER_CKPT_57 = (
    REVIEW_ROOT_57
    / "stage2_filtered_62_v1"
    / "best.pt"
)

UPDATES_57 = 1600
REPORT_57 = 400
HEAD_LR_57 = 1e-3
CLIP_57 = 5.0
SEED_57 = 5701

# ------------------------------------------------------------
# 1. Select ONLY human-reviewed training examples
# ------------------------------------------------------------

reviews_57 = []

for filename in [
    "manual_review_results_round1.json",
    "manual_review_results_round2.json"
]:
    path = REVIEW_ROOT_57 / filename

    reviews_57.extend(
        json.loads(path.read_text(encoding="utf-8"))
    )

allowed_57 = {"verified_match", "match_with_fillers"}

clean_ids_57 = [
    r["id"]
    for r in reviews_57
    if r["review_status"] in allowed_57
    and r["original_group"] in {
        "Previous 8 training",
        "New 56 training"
    }
]

assert len(clean_ids_57) == 4
assert len(set(clean_ids_57)) == 4

records_57 = {
    r["id"]: r for r in train_records
}

indices_57 = {
    r["id"]: i
    for i, r in enumerate(train_dataset.records)
}

print("FOUR-RECORDING LEARNABILITY TEST")
print("=" * 60)

for i, rid in enumerate(clean_ids_57, 1):
    rec = records_57[rid]

    print(
        f"{i}. {rid}",
        f"| {rec['duration_seconds']:.2f}s",
        f"| speaker {rec['speaker_id']}"
    )

print(
    "Distinct speakers:",
    len({
        records_57[rid]["speaker_id"]
        for rid in clean_ids_57
    })
)

# ------------------------------------------------------------
# 2. Verify CTC model configuration
# ------------------------------------------------------------

model_57 = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_57).float()

model_57.eval()
model_57.requires_grad_(False)

blank_57 = tokenizer.pad_token_id
vocab_57 = len(tokenizer)

print("\nCONFIGURATION CHECK")
print("=" * 60)

print("Tokenizer blank ID:", blank_57)
print("Model blank ID:", model_57.config.pad_token_id)
print("Tokenizer vocabulary:", vocab_57)
print("Model vocabulary:", model_57.config.vocab_size)
print("CTC reduction:", model_57.config.ctc_loss_reduction)

assert model_57.config.pad_token_id == blank_57, (
    "CRITICAL: CTC blank ID mismatch."
)

assert model_57.config.vocab_size == vocab_57, (
    "CRITICAL: Model/tokenizer vocabulary mismatch."
)

# ------------------------------------------------------------
# 3. Build exact training batches
# ------------------------------------------------------------

batches_57 = {}
references_57 = {}

for rid in clean_ids_57:

    example = train_dataset[indices_57[rid]]
    batch = ctc_collator([example])

    targets = batch["labels"][0]
    targets = targets[targets >= 0].long()

    reference = normalize_amharic_ctc(
        records_57[rid]["transcription"]
    )

    reconstructed = normalize_amharic_ctc(
        tokenizer.decode(
            targets.tolist(),
            group_tokens=False
        )
    )

    assert reconstructed == reference
    assert not (targets == blank_57).any()

    batches_57[rid] = batch
    references_57[rid] = reference

# ------------------------------------------------------------
# 4. Cache original and adapted encoder outputs
# ------------------------------------------------------------

encoder_checkpoint_57 = torch.load(
    ENCODER_CKPT_57,
    map_location="cpu",
    weights_only=True
)

feature_sets_57 = {}

for branch in ["Original pretrained", "Adapted encoder"]:

    if branch == "Adapted encoder":

        for layer, weights in zip(
            model_57.wav2vec2.encoder.layers[-2:],
            encoder_checkpoint_57["upper_layers"]
        ):
            layer.load_state_dict(
                weights,
                strict=True
            )

    model_57.eval()

    features = {}

    with torch.no_grad():

        for rid in clean_ids_57:

            batch = batches_57[rid]

            hidden = model_57.wav2vec2(
                input_values=batch[
                    "input_values"
                ].to(DEVICE_57),
                attention_mask=batch[
                    "attention_mask"
                ].to(DEVICE_57)
            ).last_hidden_state

            targets = batch["labels"][0]
            targets = targets[targets >= 0].long()

            repeated = int(
                (targets[1:] == targets[:-1])
                .sum().item()
            )

            assert hidden.shape[1] >= (
                len(targets) + repeated
            )

            assert torch.isfinite(hidden).all().item()

            features[rid] = {
                "hidden": hidden.cpu().float().contiguous(),
                "targets": targets.cpu().contiguous(),
                "frames": int(hidden.shape[1]),
                "label_count": int(targets.numel())
            }

    feature_sets_57[branch] = features
    print("\nCached:", branch)

del model_57
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5. Identical fresh head initialization
# ------------------------------------------------------------

torch.manual_seed(SEED_57)

template_head_57 = torch.nn.Linear(
    1024,
    vocab_57,
    bias=True
)

initial_head_57 = copy.deepcopy(
    template_head_57.state_dict()
)

del template_head_57

# ------------------------------------------------------------
# 6. Training and evaluation helpers
# ------------------------------------------------------------

def loss_57(head, features, rid):

    item = features[rid]

    hidden = item["hidden"].to(DEVICE_57)
    targets = item["targets"].to(DEVICE_57)

    logits = head(hidden)

    log_probs = F.log_softmax(
        logits.float(),
        dim=-1
    ).transpose(0, 1)

    return F.ctc_loss(
        log_probs,
        targets,
        torch.tensor(
            [item["frames"]],
            dtype=torch.long
        ),
        torch.tensor(
            [item["label_count"]],
            dtype=torch.long
        ),
        blank=blank_57,
        reduction="mean",
        zero_infinity=False
    )

def evaluate_57(head, features):

    head.eval()

    predictions = []
    references = []
    blank_count = 0
    frame_count = 0
    losses = []

    with torch.no_grad():

        for rid in clean_ids_57:

            item = features[rid]
            hidden = item["hidden"].to(DEVICE_57)

            logits = head(hidden)

            predicted_ids = logits.argmax(dim=-1)

            blank_count += int(
                (predicted_ids == blank_57).sum().item()
            )

            frame_count += predicted_ids.numel()

            prediction = processor.batch_decode(
                predicted_ids.cpu()
            )[0]

            predictions.append(
                normalize_amharic_ctc(prediction)
            )

            references.append(references_57[rid])

            losses.append(
                float(loss_57(head, features, rid).item())
            )

    return {
        "cer": float(cer(references, predictions)),
        "wer": float(wer(references, predictions)),
        "ctc_loss": statistics.mean(losses),
        "blank_fraction": blank_count / max(frame_count, 1),
        "length_ratio": (
            sum(len(p.replace(" ", "")) for p in predictions)
            / max(
                sum(len(r.replace(" ", "")) for r in references),
                1
            )
        ),
        "predictions": {
            rid: prediction
            for rid, prediction in zip(
                clean_ids_57, predictions
            )
        }
    }

def record_for_step_57(step):

    epoch = (step - 1) // len(clean_ids_57)
    position = (step - 1) % len(clean_ids_57)

    order = list(clean_ids_57)

    random.Random(SEED_57 + epoch).shuffle(order)

    return order[position]

# ------------------------------------------------------------
# 7. Run both independent branches
# ------------------------------------------------------------

results_57 = {}

for branch, features in feature_sets_57.items():

    branch_dir = RUN_57 / (
        "original" if branch == "Original pretrained"
        else "adapted"
    )

    branch_dir.mkdir(exist_ok=True)

    latest = branch_dir / "latest.pt"
    history_path = branch_dir / "history.json"

    head = torch.nn.Linear(
        1024, vocab_57
    ).to(DEVICE_57).float()

    head.load_state_dict(
        initial_head_57,
        strict=True
    )

    optimizer = torch.optim.AdamW(
        head.parameters(),
        lr=HEAD_LR_57,
        weight_decay=0.0
    )

    signature = hashlib.sha256(
        json.dumps(
            {
                "branch": branch,
                "source": str(ENCODER_CKPT_57),
                "recordings": [
                    {
                        "id": rid,
                        "transcript": references_57[rid]
                    }
                    for rid in clean_ids_57
                ],
                "seed": SEED_57,
                "lr": HEAD_LR_57,
                "clip": CLIP_57
            },
            sort_keys=True,
            ensure_ascii=False
        ).encode("utf-8")
    ).hexdigest()

    start_step = 0
    history = []

    if latest.exists():

        saved = torch.load(
            latest,
            map_location="cpu",
            weights_only=True
        )

        assert saved["signature"] == signature

        head.load_state_dict(saved["head"])
        optimizer.load_state_dict(saved["optimizer"])

        start_step = int(saved["step"])

        assert history_path.exists()

        history = json.loads(
            history_path.read_text(encoding="utf-8")
        )

    else:

        initial = evaluate_57(head, features)
        initial["step"] = 0
        history.append(initial)

    print("\n" + "=" * 65)
    print(branch.upper())
    print("=" * 65)
    print("Starting update:", start_step)

    recent_losses = []

    for step in range(start_step + 1, UPDATES_57 + 1):

        rid = record_for_step_57(step)

        head.train()
        optimizer.zero_grad(set_to_none=True)

        loss = loss_57(head, features, rid)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Nonfinite loss: {branch}, update {step}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            head.parameters(),
            max_norm=CLIP_57,
            error_if_nonfinite=True
        )

        optimizer.step()

        recent_losses.append(float(loss.detach().item()))

        if step % REPORT_57 == 0:

            metrics = evaluate_57(head, features)
            metrics["step"] = step

            metrics["mean_recent_train_loss"] = (
                statistics.mean(recent_losses)
            )

            history.append(metrics)

            print(
                f"Update {step:4d} | "
                f"CER {metrics['cer']*100:6.2f}% | "
                f"WER {metrics['wer']*100:6.2f}% | "
                f"CTC {metrics['ctc_loss']:6.3f} | "
                f"Length {metrics['length_ratio']:.2f} | "
                f"Blank {metrics['blank_fraction']*100:.1f}%"
            )

            checkpoint = {
                "signature": signature,
                "step": step,
                "head": head.state_dict(),
                "optimizer": optimizer.state_dict()
            }

            temp_checkpoint = branch_dir / "latest.pt.partial"
            torch.save(checkpoint, temp_checkpoint)
            temp_checkpoint.replace(latest)

            temp_history = branch_dir / "history.json.partial"

            temp_history.write_text(
                json.dumps(
                    history,
                    ensure_ascii=False,
                    indent=2
                ),
                encoding="utf-8"
            )

            temp_history.replace(history_path)

            recent_losses.clear()

    results_57[branch] = history

    del head, optimizer
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 8. Compare memorization ability
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FOUR-RECORDING LEARNABILITY RESULTS")
print("=" * 75)

print(
    f"{'Update':<10}"
    f"{'Original CER':<18}"
    f"{'Adapted CER':<18}"
)

original_history = {
    r["step"]: r
    for r in results_57["Original pretrained"]
}

adapted_history = {
    r["step"]: r
    for r in results_57["Adapted encoder"]
}

for step in sorted(
    set(original_history) & set(adapted_history)
):

    original_cer = original_history[step]["cer"] * 100
    adapted_cer = adapted_history[step]["cer"] * 100

    print(
        f"{step:<10}"
        f"{original_cer:<18.2f}"
        f"{adapted_cer:<18.2f}"
    )

print("\nFinal per-recording results:")

for branch in results_57:

    final = results_57[branch][-1]

    print("\n", branch)

    for rid in clean_ids_57:

        print("\nID:", rid)
        print("REFERENCE:", references_57[rid])
        print(
            "PREDICTION:",
            final["predictions"][rid] or "[EMPTY]"
        )

print("\nSaved experiment:", RUN_57)
print("Existing ASR checkpoints unchanged.")


## Original source cell 62

Position 63 in the original notebook; original code comments may use different numbering.
